# Exercise 1: Genomics Data Lake Anatomy
## BINFX410 — Chapter 10: Data Lakes, Warehouses, and Lakehouses

---

## Learning Objectives

By the end of this exercise you will be able to:
- Explain what a data lake is and how it differs from a database
- Organize genomics files using the Bronze/Silver/Gold (medallion) architecture
- Apply Hive-style partitioning to genomic data on S3
- Demonstrate schema-on-read by querying the same file in different ways
- Run a basic SQL query against raw genomic data using Amazon Athena

**Estimated time:** 45 minutes  
**Dataset:** GATK Test Data (`s3://gatk-test-data`) — WGS/exome BAMs, FASTQs, VCFs  
**AWS services:** S3, Athena, Glue Data Catalog

## Background: Why Do Genomics Labs Need Data Lakes?

A typical genomics research project generates many different file types across many different formats:

| File Type | Format | Size (per sample) | Purpose |
|---|---|---|---|
| Raw reads | FASTQ | 5–50 GB | Sequencer output |
| Aligned reads | BAM/CRAM | 5–30 GB | After alignment to reference |
| Variant calls | VCF | 100 MB–5 GB | SNPs, indels, SVs |
| Expression counts | TSV matrix | 10–500 MB | RNA-seq quantification |
| Sample metadata | JSON/TSV | KB–MB | Clinical/experimental annotations |

A **relational database** cannot efficiently store FASTQ files or BAMs. A **traditional file server** has no query capability. A **data lake** — object storage (S3) with a query engine (Athena) on top — gives you both: cheap, scalable storage AND the ability to query data without loading it into a database first.

### The Medallion Architecture

The **Bronze/Silver/Gold** pattern organizes data by how processed it is:

```
Bronze (Raw)          Silver (Cleaned)        Gold (Aggregated)
──────────────        ────────────────        ─────────────────
Raw FASTQ files  →    Aligned BAMs       →    Variant summary
Raw VCF calls         Filtered variants        Per-gene stats
Sample metadata       Standardized TSV         Analysis tables
Original format       Parquet, partitioned     Athena-optimized
```

**Key principle:** Never delete Bronze data. If your Silver transformation has a bug, you can re-process from Bronze.

## Section 1: Setup

**What this does:** This would install the required Python packages (`boto3`, `awswrangler`, `pandas`, `pyarrow`) into the current Python environment. In a real AWS environment these would typically be pre-installed on your SageMaker notebook instance or included in your Glue job's library configuration.

```python
# Install required packages (run once)
!pip install boto3 awswrangler pandas pyarrow --quiet
```

**What this does:** This initializes the AWS session by importing libraries and creating two S3 clients: a signed client (for your private bucket `binfx410-datalake-{STUDENT_ID}`) and an unsigned client (for accessing public S3 buckets without credentials). It also defines the student-specific bucket names used throughout the exercise.

```python
import boto3
import awswrangler as wr
import pandas as pd
import json
from botocore import UNSIGNED
from botocore.config import Config

# ──────────────────────────────────────────────────────────
# CONFIGURE THIS: replace with your student ID
# ──────────────────────────────────────────────────────────
STUDENT_ID = "sab032226"   # e.g., "jsmith" or "u1234567"
AWS_REGION = "us-east-1"

BUCKET = f"binfx410-datalake-{STUDENT_ID}"
ATHENA_RESULTS = f"s3://binfx410-athena-results-{STUDENT_ID}/"

# Clients
s3 = boto3.client('s3', region_name=AWS_REGION)
s3_unsigned = boto3.client('s3', region_name=AWS_REGION,
                            config=Config(signature_version=UNSIGNED))  # for public buckets

print(f"Working bucket: s3://{BUCKET}")
print(f"Athena results: {ATHENA_RESULTS}")
```

## Section 2: Exploring the GATK Test Data Bucket

The GATK Test Data bucket (`s3://gatk-test-data`) is maintained by the Broad Institute specifically for tutorials and exercises. It contains real WGS, exome, and RNA-seq data at small scale — perfect for teaching.

Let's explore what's available.

**What this does:** This would call the S3 `ListObjectsV2` API with `Delimiter='/'` on the public `gatk-test-data` bucket to enumerate its top-level prefixes (analogous to listing root directories). No data is downloaded — only metadata is returned. The unsigned client is used because this is a publicly readable bucket that requires no AWS credentials.

```python
# List top-level prefixes in the GATK test data bucket
GATK_BUCKET = "gatk-test-data"

response = s3_unsigned.list_objects_v2(
    Bucket=GATK_BUCKET,
    Delimiter='/'  # only top-level "directories"
)

print("Top-level prefixes in s3://gatk-test-data/:")
for prefix in response.get('CommonPrefixes', []):
    print(f"  {prefix['Prefix']}")
```

**What this does:** This would paginate through the `wgs_bam/` prefix of the public `gatk-test-data` S3 bucket using the `ListObjectsV2` paginator, collecting each object's key and size. The resulting DataFrame summarizes available file types and sizes, helping you decide which small index/checksum files are worth staging to your bronze zone.

```python
# Drill into the wgs_bam directory to see what's available
paginator = s3_unsigned.get_paginator('list_objects_v2')
pages = paginator.paginate(Bucket=GATK_BUCKET, Prefix='wgs_bam/')

files = []
for page in pages:
    for obj in page.get('Contents', []):
        files.append({
            'key': obj['Key'],
            'size_mb': round(obj['Size'] / 1024 / 1024, 2),
            'extension': obj['Key'].split('.')[-1]
        })

df_files = pd.DataFrame(files)
print(f"Found {len(df_files)} files in wgs_bam/")
print("\nFile types:")
print(df_files.groupby('extension')['size_mb'].agg(['count', 'sum']).round(2))
print("\nSample files:")
df_files.head(10)
```

## Section 3: Designing the Three-Zone Lake Structure

Before copying data, we design our S3 prefix structure. Think of S3 prefixes like folder paths — they're not real directories, but they behave like them.

### Our Zone Design

```
s3://binfx410-datalake-{student_id}/
│
├── bronze/                          ← Raw data, original format, never modified
│   ├── raw-reads/
│   │   └── sample=NA12878/          ← Hive-style partition by sample
│   │       └── reference=hg38/      ← Sub-partition by reference genome
│   └── variants/
│       └── source=gatk/             ← Partition by data source
│
├── silver/                          ← Cleaned, standardized, Parquet format
│   └── variants/
│       └── chromosome=chr22/        ← Partition by chromosome (natural genomics key!)
│
└── gold/                            ← Aggregated, analysis-ready, Athena-optimized
    └── variant_summary/
        └── variant_type=SNP/
```

### Why Hive-style partitioning?

When Athena sees `WHERE sample_id = 'NA12878'`, it can skip all S3 objects **not** under the `sample=NA12878/` prefix. This is called **partition pruning** and can reduce bytes scanned (and cost) by 100x.

## Section 4: Creating the S3 Bucket and Zone Structure

**What this does:** This would create the S3 bucket `binfx410-datalake-{STUDENT_ID}` in `us-east-1` and then write four empty placeholder objects to establish the bronze, silver, and gold zone prefixes. S3 has no real directory concept — these zero-byte `.keep` files create the visual folder structure that the AWS Console and lake tools expect.

```python
# Create your student data lake bucket
try:
    if AWS_REGION == 'us-east-1':
        s3.create_bucket(Bucket=BUCKET)
    else:
        s3.create_bucket(
            Bucket=BUCKET,
            CreateBucketConfiguration={'LocationConstraint': AWS_REGION}
        )
    print(f"Created bucket: s3://{BUCKET}")
except s3.exceptions.BucketAlreadyOwnedByYou:
    print(f"Bucket already exists: s3://{BUCKET}")

# Create placeholder objects to establish zone structure
# (S3 has no real directories, but prefixes create the illusion of them)
zones = [
    f"bronze/raw-reads/sample=NA12878/reference=hg38/.keep",
    f"bronze/variants/source=gatk/.keep",
    f"silver/variants/chromosome=chr22/.keep",
    f"gold/variant_summary/variant_type=SNP/.keep",
]

for key in zones:
    s3.put_object(Bucket=BUCKET, Key=key, Body=b'')
    print(f"  Created: s3://{BUCKET}/{key}")

print("\nZone structure initialized.")
```

## Section 5: Staging Data to the Bronze Zone

We'll copy a small VCF and its index from the GATK test data into our bronze zone. In a real pipeline this copy step would be automated (e.g., triggered when sequencing completes).

**What this does:** This would perform server-side S3 copy operations (`CopyObject`) from the public `gatk-test-data` bucket into your private bronze zone at `s3://binfx410-datalake-{STUDENT_ID}/bronze/raw-reads/sample=NA12878/reference=hg38/`. Because the source and destination are both in AWS, no data traverses the public internet — data stays within the AWS network, making this fast and free of egress charges.

```python
# Copy small index/checksum files from GATK test data to our bronze zone
# We use a server-side copy via boto3 (no data leaves AWS)
# Source keys verified from Section 2 listing above (wgs_bam/NA12878_20k_hg38/)

files_to_stage = [
    # (source_key, dest_key)
    ('wgs_bam/NA12878_20k_hg38/NA12878.bam.md5',
     'bronze/raw-reads/sample=NA12878/reference=hg38/NA12878.bam.md5'),
    ('wgs_bam/NA12878_20k_hg38/NA12878.bai',
     'bronze/raw-reads/sample=NA12878/reference=hg38/NA12878.bai'),
]

# Preview what's available in the hg38 BAM prefix
vcf_response = s3_unsigned.list_objects_v2(
    Bucket=GATK_BUCKET,
    Prefix='wgs_bam/NA12878_20k_hg38/',
    MaxKeys=20
)

print("Available files to stage:")
for obj in vcf_response.get('Contents', []):
    size_kb = obj['Size'] / 1024
    print(f"  {obj['Key']}  ({size_kb:.1f} KB)")

# Perform the server-side copy (no data leaves AWS)
print("\nStaging files to bronze zone:")
for src_key, dest_key in files_to_stage:
    s3.copy_object(
        CopySource={'Bucket': GATK_BUCKET, 'Key': src_key},
        Bucket=BUCKET,
        Key=dest_key
    )
    print(f"  Copied: {src_key}  →  s3://{BUCKET}/{dest_key}")
```

**What this does:** This would create a two-row sample manifest DataFrame and write it as a CSV file to `s3://binfx410-datalake-{STUDENT_ID}/bronze/raw-reads/sample_manifest.csv` using `S3.put_object`. In a real lab environment this manifest would track sample provenance, consent status, QC metrics, and instrument metadata for every sequenced individual.

```python
# Write a synthetic sample manifest as a stand-in for real metadata
# (Real labs maintain sample manifests tracking provenance, QC, consent)

manifest = pd.DataFrame([
    {
        'sample_id': 'NA12878',
        'reference_genome': 'hg38',
        'sequencing_type': 'WGS',
        'instrument': 'NovaSeq 6000',
        'read_length': 150,
        'coverage': 30,
        'collection_date': '2024-01-15',
        'population': 'CEU',
        'source': 'GATK Test Data'
    },
    {
        'sample_id': 'NA19240',
        'reference_genome': 'hg38',
        'sequencing_type': 'WGS',
        'instrument': 'NovaSeq 6000',
        'read_length': 150,
        'coverage': 30,
        'collection_date': '2024-01-18',
        'population': 'YRI',
        'source': 'GATK Test Data'
    }
])

# Write manifest as CSV to bronze zone (original format preserved)
csv_body = manifest.to_csv(index=False)
s3.put_object(
    Bucket=BUCKET,
    Key='bronze/raw-reads/sample_manifest.csv',
    Body=csv_body.encode('utf-8'),
    ContentType='text/csv'
)
print("Staged sample manifest to bronze zone:")
print(manifest.to_string(index=False))
```

## Section 6: Hive-Style Partitioning in Practice

The key insight: S3 is just a key-value store. The "path" is just part of the key name. But when we structure keys as `key=value/`, query engines like Athena recognize this as a partition and use it for optimization.

**What this does:** This would write synthetic variant call data for two samples (NA12878 and NA19240) to S3 using Hive-style partitioning: `bronze/variants/source=gatk/sample=NA12878/reference=hg38/variants.csv` and `bronze/variants/source=gatk/sample=NA19240/reference=hg38/variants.csv`. Athena and Glue use these partition key=value path components to prune which files are scanned when a query filters on `sample_id` or `reference`, reducing both query time and cost.

```python
# Simulate writing variant calls for multiple samples to illustrate partitioning
# In reality these would come from GATK HaplotypeCaller output

import io

# Create synthetic variant data for two samples
variants_na12878 = pd.DataFrame([
    {'chrom': 'chr1', 'pos': 925952,  'ref': 'G', 'alt': 'A', 'qual': 50.2, 'filter': 'PASS', 'variant_type': 'SNP'},
    {'chrom': 'chr1', 'pos': 931279,  'ref': 'A', 'alt': 'G', 'qual': 88.1, 'filter': 'PASS', 'variant_type': 'SNP'},
    {'chrom': 'chr1', 'pos': 935222,  'ref': 'C', 'alt': 'CT', 'qual': 72.3, 'filter': 'PASS', 'variant_type': 'INDEL'},
])

variants_na19240 = pd.DataFrame([
    {'chrom': 'chr1', 'pos': 925952,  'ref': 'G', 'alt': 'A', 'qual': 65.4, 'filter': 'PASS', 'variant_type': 'SNP'},
    {'chrom': 'chr1', 'pos': 948428,  'ref': 'T', 'alt': 'C', 'qual': 91.0, 'filter': 'PASS', 'variant_type': 'SNP'},
])

# Write with Hive-style partitioning: sample=NA12878/reference=hg38/
for sample_id, df in [('NA12878', variants_na12878), ('NA19240', variants_na19240)]:
    key = f"bronze/variants/source=gatk/sample={sample_id}/reference=hg38/variants.csv"
    body = df.to_csv(index=False).encode()
    s3.put_object(Bucket=BUCKET, Key=key, Body=body)
    print(f"Written: s3://{BUCKET}/{key}  ({len(df)} rows)")

print("\nPartition structure allows Athena to skip NA19240 data when WHERE sample='NA12878'")
```

## Section 7: Schema-on-Read Demo

**Schema-on-read** means we define the structure when we query, not when we store. The same S3 file looks completely different depending on which tool reads it.

Let's demonstrate with a VCF-like file:

**What this does:** This would write a five-variant VCF file (VCFv4.2 format with `##` meta-headers and a `#CHROM` column header line) to `s3://binfx410-datalake-{STUDENT_ID}/bronze/variants/source=synthetic/sample=NA12878/reference=hg38/NA12878.vcf` using `S3.put_object`. This single object can then be read back in three different ways to demonstrate schema-on-read.

```python
# Write a minimal VCF-formatted file to S3
vcf_content = """##fileformat=VCFv4.2
##FILTER=<ID=PASS,Description="All filters passed">
##INFO=<ID=DP,Number=1,Type=Integer,Description="Total Depth">
##INFO=<ID=AF,Number=A,Type=Float,Description="Allele Frequency">
#CHROM\tPOS\tID\tREF\tALT\tQUAL\tFILTER\tINFO
chr1\t925952\t.\tG\tA\t50.2\tPASS\tDP=42;AF=0.45
chr1\t931279\t.\tA\tG\t88.1\tPASS\tDP=67;AF=0.52
chr1\t935222\t.\tC\tCT\t72.3\tPASS\tDP=38;AF=0.28
chr22\t16062\t.\tC\tT\t95.0\tPASS\tDP=110;AF=0.50
chr22\t16150\t.\tG\tA\t88.4\tPASS\tDP=98;AF=0.48
"""

s3.put_object(
    Bucket=BUCKET,
    Key='bronze/variants/source=synthetic/sample=NA12878/reference=hg38/NA12878.vcf',
    Body=vcf_content.encode()
)
print("VCF file written to S3.")
```

**What this does:** This would call `S3.get_object` to download the VCF file from your bronze zone and decode it as a raw UTF-8 string — exactly how a command-line tool like `samtools view` or `bcftools` would see it. No schema is applied; the `##` header lines and tab-delimited data rows are returned as-is.

```python
# Way 1: Read as raw text (what samtools / bcftools sees)
obj = s3.get_object(Bucket=BUCKET,
                    Key='bronze/variants/source=synthetic/sample=NA12878/reference=hg38/NA12878.vcf')
raw_text = obj['Body'].read().decode()

print("=== Raw text (schema-less) ===")
print(raw_text)
```

**What this does:** This would parse the same VCF S3 object into a pandas DataFrame by filtering out the `##` meta-header lines and using `pd.read_csv` with a tab separator. The `#CHROM` column header is cleaned to `CHROM`. This demonstrates schema-on-read: the same bytes stored in S3 are now viewed as a structured table without any ETL step.

```python
# Way 2: Read as structured DataFrame (skip ## header lines)
import io

lines = [l for l in raw_text.split('\n') if l and not l.startswith('##')]
vcf_df = pd.read_csv(io.StringIO('\n'.join(lines)), sep='\t')
vcf_df.columns = [c.lstrip('#') for c in vcf_df.columns]  # remove leading #

print("=== As DataFrame (with schema applied at read time) ===")
print(vcf_df.to_string(index=False))
```

**What this does:** This would further transform the DataFrame by splitting the semicolon-delimited `INFO` field (e.g., `DP=42;AF=0.45`) into separate typed columns (`DP`, `AF`). All three cells (raw text, DataFrame, structured DataFrame) read the same S3 object — illustrating that schema-on-read lets you impose different schemas on the same data depending on the analytic tool being used.

```python
# Way 3: Parse the INFO field into structured columns
def parse_info(info_str):
    result = {}
    for field in info_str.split(';'):
        if '=' in field:
            k, v = field.split('=', 1)
            try:
                result[k] = float(v)
            except ValueError:
                result[k] = v
    return result

info_expanded = vcf_df['INFO'].apply(parse_info).apply(pd.Series)
vcf_structured = pd.concat([vcf_df.drop('INFO', axis=1), info_expanded], axis=1)

print("=== Fully parsed with INFO fields extracted ===")
print(vcf_structured.to_string(index=False))

print("\nKey insight: All three representations point to the SAME S3 object.")
print("Schema-on-read means the structure is applied at query time, not storage time.")
```

## Section 8: Querying with Amazon Athena

Now let's register our staged CSV variant data as an Athena table and run SQL against it — without loading it into any database.

**What this does:** This would call the AWS Glue `CreateDatabase` API to create a database named `binfx410_bronze` in the Glue Data Catalog for your account and region. The Glue Data Catalog is the centralized metadata store that Athena, Glue ETL jobs, and Lake Formation all use to discover tables — creating this database is the prerequisite for registering any Athena-queryable tables.

```python
# Create a Glue database for our bronze zone
glue = boto3.client('glue', region_name=AWS_REGION)

try:
    glue.create_database(DatabaseInput={'Name': 'binfx410_bronze'})
    print("Created Glue database: binfx410_bronze")
except glue.exceptions.AlreadyExistsException:
    print("Database binfx410_bronze already exists.")
```

**What this does:** This would read back one of the staged variant CSV files from `s3://binfx410-datalake-{STUDENT_ID}/bronze/variants/source=gatk/sample=NA12878/reference=hg38/variants.csv` and print its pandas schema. This schema inspection step informs the column definitions that `awswrangler` will write into the Glue Data Catalog table in the next cell.

```python
# Register the variant CSV files as an Athena table using awswrangler
# awswrangler handles the Glue table creation under the hood

# First, let's read one of our staged CSVs to inspect the schema
obj = s3.get_object(Bucket=BUCKET, Key='bronze/variants/source=gatk/sample=NA12878/reference=hg38/variants.csv')
sample_df = pd.read_csv(io.BytesIO(obj['Body'].read()))
print("Schema of our variant CSV:")
print(sample_df.dtypes)
print()
print(sample_df)
```

**What this does:** This would combine the two sample variant DataFrames and use `awswrangler.s3.to_csv` with `dataset=True` to write partitioned CSV files to `s3://binfx410-datalake-{STUDENT_ID}/bronze/variants/source=gatk/` and simultaneously register an external Glue table named `binfx410_bronze.gatk_variants` with `sample_id` as a partition column. After this call, Athena can query the table immediately without needing a crawler.

```python
# Combine both samples and write as a partitioned dataset for Athena
# awswrangler will handle creating the Glue table automatically

all_variants = pd.concat([
    variants_na12878.assign(sample_id='NA12878', reference='hg38'),
    variants_na19240.assign(sample_id='NA19240', reference='hg38')
], ignore_index=True)

result = wr.s3.to_csv(
    df=all_variants,
    path=f"s3://{BUCKET}/bronze/variants/source=gatk/",
    dataset=True,
    database='binfx410_bronze',
    table='gatk_variants',
    partition_cols=['sample_id'],
    boto3_session=boto3.Session(region_name=AWS_REGION),
    index=False
)
print("Table registered in Glue catalog:")
print(f"  Database: binfx410_bronze")
print(f"  Table: gatk_variants")
print(f"  Location: s3://{BUCKET}/bronze/variants/source=gatk/")
```

**What this does:** This would execute an Athena SQL query that groups all rows in `binfx410_bronze.gatk_variants` by `sample_id`, `chrom`, and `variant_type`, computing the count and average quality score per group. Athena charges $5.00/TB scanned; because this is a small CSV dataset the cost would be a fraction of a cent. Results are written to `s3://binfx410-athena-results-{STUDENT_ID}/` and returned as a pandas DataFrame.

```python
# Query with Athena — SQL against files in S3, no database required!
query = """
SELECT
    sample_id,
    chrom,
    variant_type,
    COUNT(*) AS variant_count,
    AVG(qual) AS avg_quality
FROM binfx410_bronze.gatk_variants
GROUP BY sample_id, chrom, variant_type
ORDER BY sample_id, chrom
"""

df_result = wr.athena.read_sql_query(
    sql=query,
    database='binfx410_bronze',
    s3_output=ATHENA_RESULTS,
    boto3_session=boto3.Session(region_name=AWS_REGION)
)

print("Athena query result:")
print(df_result.to_string(index=False))
```

**What this does:** This would execute an Athena SQL query with `WHERE sample_id = 'NA12878'`, which triggers Athena's partition pruning: Athena reads the Glue Data Catalog partition metadata and skips the `sample=NA19240/` S3 prefix entirely, scanning only the NA12878 partition files. This is a key cost-saving feature of Hive-style partitioned datasets — you pay only for the data Athena actually reads.

```python
# Demonstrate partition pruning — filter to one sample only
# Athena will only scan the sample=NA12878/ prefix, skipping NA19240

pruned_query = """
SELECT chrom, pos, ref, alt, qual
FROM binfx410_bronze.gatk_variants
WHERE sample_id = 'NA12878'  -- Athena prunes: only scans sample=NA12878/ partition
ORDER BY chrom, pos
"""

df_pruned = wr.athena.read_sql_query(
    sql=pruned_query,
    database='binfx410_bronze',
    s3_output=ATHENA_RESULTS,
    boto3_session=boto3.Session(region_name=AWS_REGION)
)

print("Query result (NA12878 only):")
print(df_pruned.to_string(index=False))
```

## Section 9: Verify Your Lake Structure

**What this does:** This would paginate through all objects in `s3://binfx410-datalake-{STUDENT_ID}/` using `ListObjectsV2` and organize them into a DataFrame grouped by zone prefix (bronze, silver, gold). This gives a structural overview of the data lake's contents, showing each file's S3 key and byte size, so you can verify the zone layout matches the intended medallion architecture.

```python
# List all objects in your data lake bucket to confirm the structure
paginator = s3.get_paginator('list_objects_v2')
pages = paginator.paginate(Bucket=BUCKET)

all_objects = []
for page in pages:
    for obj in page.get('Contents', []):
        if not obj['Key'].endswith('.keep'):  # skip placeholder files
            all_objects.append({'key': obj['Key'], 'size_bytes': obj['Size']})

df_lake = pd.DataFrame(all_objects)
df_lake['zone'] = df_lake['key'].str.split('/').str[0]

print(f"Your data lake: s3://{BUCKET}/")
print(f"Total objects: {len(df_lake)}")
print()
for zone, group in df_lake.groupby('zone'):
    print(f"  {zone}/")
    for _, row in group.iterrows():
        print(f"    {row['key']}  ({row['size_bytes']} bytes)")
```

## Exercise: YOUR CODE HERE

Complete the following tasks:

**What this does:** This is a student exercise scaffold. When completed, it would write variant data for a third sample (`HG00096`) with at least three chromosome 22 variants to `s3://binfx410-datalake-{STUDENT_ID}/bronze/variants/source=gatk/sample=HG00096/reference=hg38/` using the same Hive-style partitioning pattern established in Section 6.

```python
# TASK 1: Add a third sample to the bronze zone
# Create variant data for sample 'HG00096' with at least 3 variants on chr22
# Write it to the correct partitioned location: bronze/variants/source=gatk/sample=HG00096/reference=hg38/

# YOUR CODE HERE
variants_hg00096 = pd.DataFrame([])

# Write to S3 with correct partitioning
# ...
```

**What this does:** This is a student exercise scaffold. When completed, it would transform the sample manifest from the bronze zone — adding a `processed_date` column and renaming `population` to `superpopulation_code` — and write the result as a Parquet file to `s3://binfx410-datalake-{STUDENT_ID}/silver/metadata/sample_manifest.parquet`. Writing to the silver zone as Parquet represents the Bronze→Silver data quality promotion in the medallion architecture.

```python
# TASK 2: Write the sample manifest to the Silver zone as Parquet
# The Silver version should:
#   - Add a 'processed_date' column with today's date
#   - Rename 'population' to 'superpopulation_code'
#   - Be saved as Parquet (not CSV) to s3://{BUCKET}/silver/metadata/sample_manifest.parquet

# YOUR CODE HERE
silver_manifest = manifest.copy()
# ...
```

**What this does:** This is a student exercise scaffold. When completed, it would execute an Athena SQL query against `binfx410_bronze.gatk_variants` to find which sample has the highest average variant quality score, demonstrating how to surface analytic insights from the structured data lake using standard SQL.

```python
# TASK 3: Write an Athena query that answers:
# "Which sample has the highest average variant quality score?"

quality_query = """
-- YOUR SQL HERE
"""

# df_quality = wr.athena.read_sql_query(sql=quality_query, database='binfx410_bronze', s3_output=ATHENA_RESULTS)
# print(df_quality)
```

## Reflection Questions

Answer the following questions in the cell below (double-click to edit).

1. **Why can't you store a BAM file in a relational database like PostgreSQL?** What specifically about the BAM format makes object storage (S3) a better fit?

2. **The Bronze zone keeps data in original formats (VCF, FASTQ, BAM).** If your Silver ETL job has a bug that corrupts 10,000 files, how does the Bronze zone protect you? What would you do without it?

3. **You partitioned variants by `sample_id`.** A colleague suggests also partitioning by `chromosome`. What query patterns would benefit from a chromosome partition? Can you think of a query that would actually be *slower* with chromosome partitioning?

4. **Schema-on-read vs schema-on-write:** A clinical lab insists that patient variant data must conform to a strict schema (ACMG classification fields required, no missing values). Is a data lake appropriate for their use case? What would you recommend?

5. **In the Athena query, you used `GROUP BY sample_id, chrom, variant_type`.** Without partition pruning, how many S3 objects would Athena scan? With pruning on `sample_id = 'NA12878'`, how many would it scan? (Look at your lake structure output.)

*(Double-click here to write your answers)*

1. 

2. 

3. 

4. 

5. 